# Palace Field Visualization

Top-view visualization of electromagnetic fields from a Palace driven
simulation on a CPW (coplanar waveguide) structure at ~10 GHz.

**Requirements:** `uv pip install pyvista matplotlib scipy`

## Simulation setup (reference)

The simulation was configured and run with the code below. Results are
pre-computed in `sim-data-palace-a2a1bbd0/`.

In [ ]:
import gdsfactory as gf
from ihp import LAYER, PDK
from gsim.palace import DrivenSim

PDK.activate()


@gf.cell
def gsg_electrode(
    length=300, s_width=20, g_width=40, gap_width=15, layer=LAYER.TopMetal2drawing
):
    c = gf.Component()
    r1 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r1.move((0, (g_width + s_width) / 2 + gap_width))
    _r2 = c << gf.c.rectangle((length, s_width), centered=True, layer=layer)
    r3 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r3.move((0, -(g_width + s_width) / 2 - gap_width))
    c.add_port(
        name="o1",
        center=(-length / 2, 0),
        width=s_width,
        orientation=0,
        port_type="electrical",
        layer=layer,
    )
    c.add_port(
        name="o2",
        center=(length / 2, 0),
        width=s_width,
        orientation=180,
        port_type="electrical",
        layer=layer,
    )
    return c


sim = DrivenSim()
sim.set_output_dir("./palace-sim-cpw-fields")
sim.set_geometry(gsg_electrode())
sim.set_stack(substrate_thickness=2.0, air_above=300.0)
sim.add_cpw_port("o1", layer="topmetal2", s_width=20, gap_width=15)
sim.add_cpw_port("o2", layer="topmetal2", s_width=20, gap_width=15)

# Single frequency point at 50 GHz, adaptive off so Palace does a full solve
sim.set_driven(
    fmin=50e9,
    fmax=50e9 + 1e6,  # tiny range = effectively one point
    num_points=1,
    adaptive_tol=0,
    save_step=1,
)
sim.mesh(preset="fine", planar_conductors=False)
results = sim.run()

## Load results

In [ ]:
from __future__ import annotations

from pathlib import Path
from xml.etree import ElementTree

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from matplotlib.colors import LogNorm
from scipy.interpolate import griddata

pv.OFF_SCREEN = True

# Get results dir from sim output (or hardcode for re-runs)
results_dir = Path(results["port-S.csv"]).parents[3]
freq_ghz = 50.0  # frequency we simulated at
print(f"Results dir: {results_dir}")

vol_dir = results_dir / "results/palace/paraview/driven/excitation_1"
bnd_dir = results_dir / "results/palace/paraview/driven_boundary/excitation_1"

# Use the last .pvtu (final solve iteration)
vol_path = sorted(vol_dir.rglob("*.pvtu"))[-1]
bnd_path = sorted(bnd_dir.rglob("*.pvtu"))[-1]

vol = pv.read(str(vol_path))
bnd = pv.read(str(bnd_path))

print(f"Frequency: {freq_ghz:.1f} GHz")
print(f"Volume: {vol.n_points:,} points, {vol.n_cells:,} cells")
print(f"Boundary: {bnd.n_points:,} points, {bnd.n_cells:,} cells")

In [ ]:
## Top view at conductor layer

In [ ]:
# Slice volume at conductor top, clip boundary to conductor region
z_conductor = 16.3
vol_slice = vol.slice(normal="z", origin=(0, 0, z_conductor))
bnd_clip = bnd.clip(normal="z", origin=(0, 0, 17), invert=True)

# Fit interpolation grid to the actual data bounds
vpts_all = vol_slice.points
x_pad, y_pad = 5, 5
xi = np.linspace(vpts_all[:, 0].min() - x_pad, vpts_all[:, 0].max() + x_pad, 500)
yi = np.linspace(vpts_all[:, 1].min() - y_pad, vpts_all[:, 1].max() + y_pad, 150)
Xi, Yi = np.meshgrid(xi, yi)

FONTSIZE = 11


def interp2d(points, values, grid_x, grid_y):
    return griddata(points, values, (grid_x, grid_y), method="linear")


def plot_topview(pts, data, title, cmap="turbo", scale="linear"):
    gi = interp2d(pts[:, :2], data, Xi, Yi)
    fig, ax = plt.subplots(figsize=(14, 3.5))
    if scale == "linear":
        vmax = np.nanpercentile(gi, 98)
        im = ax.pcolormesh(Xi, Yi, gi, cmap=cmap, vmin=0, vmax=vmax, shading="auto")
    elif scale == "sym":
        vmax = np.nanpercentile(np.abs(gi), 98)
        im = ax.pcolormesh(Xi, Yi, gi, cmap=cmap, vmin=-vmax, vmax=vmax, shading="auto")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlabel("x (\u00b5m)")
    ax.set_ylabel("y (\u00b5m)")
    ax.tick_params()
    valid = ~np.isnan(gi)
    if valid.any():
        rows = np.any(valid, axis=1)
        cols = np.any(valid, axis=0)
        ax.set_xlim(xi[cols][0], xi[cols][-1])
        ax.set_ylim(yi[rows][0], yi[rows][-1])
    fig.tight_layout(pad=0.5)
    plt.show()


vpts = vol_slice.points
bpts = bnd_clip.points

In [ ]:
plot_topview(
    vpts,
    np.linalg.norm(vol_slice.point_data["E_real"], axis=1),
    f"Electric field |E| at {freq_ghz:.1f} GHz \u2014 field concentrated in CPW gaps (V/m)",
)

In [ ]:
plot_topview(
    vpts,
    np.linalg.norm(vol_slice.point_data["S"], axis=1),
    f"Poynting vector |S| at {freq_ghz:.1f} GHz \u2014 power flow along the waveguide (W/m\u00b2)",
)

In [ ]:
# Surface current density — nearest interpolation
js_mag = np.linalg.norm(bnd_clip.point_data["J_s_real"], axis=1)
gi = griddata(bpts[:, :2], js_mag, (Xi, Yi), method="nearest")

fig, ax = plt.subplots(figsize=(14, 3.5))
vmax = np.nanpercentile(gi, 98)
im = ax.pcolormesh(Xi, Yi, gi, cmap="turbo", vmin=0, vmax=vmax, shading="auto")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
ax.set_title(
    f"Surface current |J_s| at {freq_ghz:.1f} GHz \u2014 current crowding at conductor edges (A/m)"
)
ax.set_aspect("equal")
ax.set_xlabel("x (\u00b5m)")
ax.set_ylabel("y (\u00b5m)")
ax.tick_params()
valid = gi > 0
if valid.any():
    rows = np.any(valid, axis=1)
    cols = np.any(valid, axis=0)
    ax.set_xlim(xi[cols][0], xi[cols][-1])
    ax.set_ylim(yi[rows][0], yi[rows][-1])
fig.tight_layout(pad=0.5)
plt.show()

## Cross-sections (YZ plane at x=0)

In [ ]:
# YZ cross-section at x=0 with E-field vector arrows
from mpl_toolkits.axes_grid1 import make_axes_locatable

yz_slice = vol.slice(normal="x", origin=(0, 0, 0))
yz_pts = yz_slice.points
E_yz = yz_slice.point_data["E_real"]
E_yz_mag = np.linalg.norm(E_yz, axis=1)

y_pad = 5
yi_cs = np.linspace(yz_pts[:, 1].min() - y_pad, yz_pts[:, 1].max() + y_pad, 200)
zi_cs = np.linspace(-5, 50, 100)
Yi_cs, Zi_cs = np.meshgrid(yi_cs, zi_cs)

E_mag_cs = interp2d(yz_pts[:, 1:3], E_yz_mag, Yi_cs, Zi_cs)
Ey_cs = interp2d(yz_pts[:, 1:3], E_yz[:, 1], Yi_cs, Zi_cs)
Ez_cs = interp2d(yz_pts[:, 1:3], E_yz[:, 2], Yi_cs, Zi_cs)

fig, ax = plt.subplots(figsize=(12, 5))
vmax_cs = np.nanpercentile(E_mag_cs, 98)
im = ax.pcolormesh(
    Yi_cs, Zi_cs, E_mag_cs, cmap="turbo", vmin=0, vmax=vmax_cs, shading="auto"
)

skip = 8
ax.quiver(
    Yi_cs[::skip, ::skip],
    Zi_cs[::skip, ::skip],
    Ey_cs[::skip, ::skip],
    Ez_cs[::skip, ::skip],
    color="white",
    alpha=0.7,
    scale=vmax_cs * 15,
    width=0.002,
)
ax.set_title(
    f"E-field cross-section at {freq_ghz:.1f} GHz \u2014 field lines from signal to ground",
)
ax.set_xlabel("y (\u00b5m)")
ax.set_ylabel("z (\u00b5m)")
ax.tick_params()
ax.set_aspect("equal")
valid = ~np.isnan(E_mag_cs)
if valid.any():
    rows = np.any(valid, axis=1)
    cols = np.any(valid, axis=0)
    ax.set_xlim(yi_cs[cols][0], yi_cs[cols][-1])
    ax.set_ylim(zi_cs[rows][0], zi_cs[rows][-1])
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="2%", pad=0.1)
fig.colorbar(im, cax=cax, label="|E| (V/m)")
fig.tight_layout(pad=0.5)
plt.show()

In [ ]:
# YZ cross-section at x=0 with H-field (B) vector arrows
B_yz = yz_slice.point_data["B_real"]
B_yz_mag = np.linalg.norm(B_yz, axis=1)

B_mag_cs = interp2d(yz_pts[:, 1:3], B_yz_mag, Yi_cs, Zi_cs)
By_cs = interp2d(yz_pts[:, 1:3], B_yz[:, 1], Yi_cs, Zi_cs)
Bz_cs = interp2d(yz_pts[:, 1:3], B_yz[:, 2], Yi_cs, Zi_cs)

fig, ax = plt.subplots(figsize=(12, 5))
vmax_b = np.nanpercentile(B_mag_cs, 98)
im = ax.pcolormesh(
    Yi_cs, Zi_cs, B_mag_cs, cmap="turbo", vmin=0, vmax=vmax_b, shading="auto"
)

skip = 8
ax.quiver(
    Yi_cs[::skip, ::skip],
    Zi_cs[::skip, ::skip],
    By_cs[::skip, ::skip],
    Bz_cs[::skip, ::skip],
    color="white",
    alpha=0.7,
    scale=vmax_b * 15,
    width=0.002,
)
ax.set_title(
    f"H-field cross-section at {freq_ghz:.1f} GHz \u2014 magnetic field circulating around conductor",
)
ax.set_xlabel("y (\u00b5m)")
ax.set_ylabel("z (\u00b5m)")
ax.tick_params()
ax.set_aspect("equal")
valid = ~np.isnan(B_mag_cs)
if valid.any():
    rows = np.any(valid, axis=1)
    cols = np.any(valid, axis=0)
    ax.set_xlim(yi_cs[cols][0], yi_cs[cols][-1])
    ax.set_ylim(zi_cs[rows][0], zi_cs[rows][-1])
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="2%", pad=0.1)
fig.colorbar(im, cax=cax, label="|B| (T)")
fig.tight_layout(pad=0.5)
plt.show()